# 02 - Huấn luyện các thực nghiệm

E0 SimpleCNN, E1 ResNet-18 frozen, E2 ResNet-18 fine-tuning và E3 fine-tuning + augmentation. Chọn checkpoint theo validation Macro-F1.

In [2]:
from pathlib import Path
import sys
import os
import traceback
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import make_dataloaders, seed_everything
from src.model import SimpleCNN, build_resnet18
from src.train import ExperimentLogger, fit

try:
    import wandb
except ImportError:
    wandb = None

SEED, BATCH_SIZE, EPOCHS = 42, 32, 10
MAX_ATTEMPTS = 3
WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'plant-disease-deep-learning')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
metadata = pd.read_csv(PROJECT_ROOT / 'data' / 'metadata.csv')
NUM_CLASSES = metadata['label'].nunique()

print('Device:', DEVICE, 'classes:', NUM_CLASSES)
print('W&B project:', WANDB_PROJECT)
print('W&B package:', 'ready' if wandb is not None else 'missing - pip install wandb')

Device: cpu classes: 38
W&B project: plant-disease-deep-learning
W&B package: ready


In [ ]:
experiments = [
    ('E0_simplecnn', lambda: SimpleCNN(NUM_CLASSES), False, 1e-3),
    ('E1_resnet_frozen', lambda: build_resnet18(NUM_CLASSES, freeze_backbone=True), False, 1e-3),
    ('E2_resnet_finetune', lambda: build_resnet18(NUM_CLASSES, freeze_backbone=False), False, 1e-4),
    ('E3_resnet_finetune_aug', lambda: build_resnet18(NUM_CLASSES, freeze_backbone=False), True, 1e-4),
]
PROJECT_ROOT.joinpath('outputs').mkdir(exist_ok=True)
RUN_SEEDS = [42, 43, 44, 45, 46]
logger = ExperimentLogger(PROJECT_ROOT / 'outputs' / 'experiment_log.jsonl')
histories = {}
models = {}

for name, factory, augmentation, lr in experiments:
    for run_seed in RUN_SEEDS:
        run_id = f'{name}_seed{run_seed}'

        for attempt in range(1, MAX_ATTEMPTS + 1):
            attempt_id = f'{run_id}_attempt{attempt}'
            print(f'\n=== {attempt_id} ===')
            wandb_run = None

            if wandb is not None:
                try:
                    wandb_run = wandb.init(
                        project=WANDB_PROJECT,
                        name=attempt_id,
                        group=run_id,
                        job_type='training',
                        reinit='finish_previous',
                        tags=[name, f'seed-{run_seed}', 'experiment', f'attempt-{attempt}'],
                        config={
                            'experiment': name,
                            'seed': run_seed,
                            'batch_size': BATCH_SIZE,
                            'epochs': EPOCHS,
                            'learning_rate': lr,
                            'augmentation': augmentation,
                            'num_classes': NUM_CLASSES,
                            'device': str(DEVICE),
                            'attempt': attempt,
                            'max_attempts': MAX_ATTEMPTS,
                        },
                    )
                    wandb_run.define_metric('epoch')
                    wandb_run.define_metric('*', step_metric='epoch')
                except Exception as wandb_error:
                    logger.log(
                        event='wandb_init_failed', run_id=run_id,
                        experiment=name, attempt=attempt,
                        error_type=type(wandb_error).__name__,
                        error_message=str(wandb_error), traceback=traceback.format_exc(),
                    )
                    print(f'W&B khong khoi dong duoc: {wandb_error}')

            try:
                seed_everything(run_seed)
                loaders = make_dataloaders(metadata, BATCH_SIZE, augmentation=augmentation)
                model, history = fit(
                    model=factory(), loaders=loaders, epochs=EPOCHS, device=DEVICE,
                    learning_rate=lr,
                    checkpoint_path=PROJECT_ROOT / 'checkpoints' / f'{run_id}.pt',
                    logger=logger, run_id=run_id, experiment_name=name,
                    wandb_run=wandb_run,
                )

                history_df = pd.DataFrame(history)
                histories[run_id] = history_df
                models[run_id] = model.cpu()
                history_df.to_csv(PROJECT_ROOT / 'outputs' / f'{run_id}_history.csv', index=False)

                if wandb_run is not None:
                    wandb_run.summary['status'] = 'success'
                    wandb_run.summary['attempt'] = attempt
                    wandb_run.save(str(PROJECT_ROOT / 'outputs' / f'{run_id}_history.csv'))
                print(f'Thanh cong: {run_id}')
                break

            except Exception as error:
                error_text = traceback.format_exc()
                logger.log(
                    event='run_attempt_failed', run_id=run_id,
                    experiment=name, attempt=attempt,
                    error_type=type(error).__name__, error_message=str(error),
                    traceback=error_text,
                )

                if wandb_run is not None:
                    wandb_run.log({
                        'status/run_failed': 1,
                        'error/error_type': type(error).__name__,
                        'error/error_message': str(error),
                    })
                    wandb_run.summary['status'] = 'failed'
                    wandb_run.summary['error_type'] = type(error).__name__
                    wandb_run.summary['error_message'] = str(error)
                    wandb_run.summary['traceback'] = error_text

                print(f'That bai lan {attempt}/{MAX_ATTEMPTS}: {error}')

                if attempt == MAX_ATTEMPTS:
                    print(f'Dung retry sau {MAX_ATTEMPTS} lan cho {run_id}')

            finally:
                if wandb_run is not None:
                    wandb_run.finish(exit_code=0 if histories.get(run_id) is not None else 1)

In [ ]:
import matplotlib.pyplot as plt
PROJECT_ROOT.joinpath('outputs').mkdir(exist_ok=True)
for name, history in histories.items():
    plt.plot(history['epoch'], history['val_macro_f1'], label=name)
plt.xlabel('Epoch'); plt.ylabel('Validation Macro-F1'); plt.legend(); plt.grid(alpha=.2)
plt.tight_layout(); plt.savefig(PROJECT_ROOT / 'outputs' / 'validation_f1_comparison.png', dpi=160); plt.show()